In [1]:
import os
import shutil
import random
from PIL import Image

# =========================
# CONFIG
# =========================
BASE_DIR = "/kaggle/input/datasets/freedomfighter1290/wheat-disease/Wheat_Disease"
TARGET_DIR = "/kaggle/working/wheat_final"

SELECTED_CLASSES = {
    "Healthy Wheat": "healthy",
    "Mildew": "mildew",
    "Yellow Rust": "yellow_rust",
    "Brown Rust": "brown_rust"
}

IMG_SIZE = (224, 224)
BALANCE_LIMIT = 1500   # set None to disable

random.seed(42)

# =========================
# CREATE OUTPUT FOLDERS
# =========================
for split in ["train", "val", "test"]:
    for cls in SELECTED_CLASSES.values():
        os.makedirs(os.path.join(TARGET_DIR, split, cls), exist_ok=True)

# =========================
# COLLECT DATA FROM BOTH
# =========================
all_images = {cls: [] for cls in SELECTED_CLASSES.keys()}

for split in ["train", "validation"]:
    for cls in SELECTED_CLASSES.keys():

        path = os.path.join(BASE_DIR, split, cls)

        if not os.path.exists(path):
            continue

        for img in os.listdir(path):
            all_images[cls].append(os.path.join(path, img))

# =========================
# PROCESS EACH CLASS
# =========================
for original_cls, new_cls in SELECTED_CLASSES.items():

    print(f"\nProcessing {original_cls}")

    images = all_images[original_cls]
    print("Total collected:", len(images))

    cleaned = []

    # -------------------------
    # CLEAN + STANDARDIZE
    # -------------------------
    for img_path in images:
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(IMG_SIZE)
            cleaned.append(img)
        except:
            continue

    print("Valid images:", len(cleaned))

    # -------------------------
    # BALANCE (optional)
    # -------------------------
    if BALANCE_LIMIT:
        cleaned = random.sample(cleaned, min(len(cleaned), BALANCE_LIMIT))

    print("After balancing:", len(cleaned))

    # -------------------------
    # SPLIT (80/10/10)
    # -------------------------
    random.shuffle(cleaned)

    n = len(cleaned)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)

    splits = {
        "train": cleaned[:train_end],
        "val": cleaned[train_end:val_end],
        "test": cleaned[val_end:]
    }

    # -------------------------
    # SAVE
    # -------------------------
    for split, imgs in splits.items():
        for i, img in enumerate(imgs):
            save_path = os.path.join(
                TARGET_DIR,
                split,
                new_cls,
                f"{new_cls}_{i}.jpg"
            )
            img.save(save_path, "JPEG", quality=95)

print("\n✅ FINAL DATASET READY:", TARGET_DIR)


Processing Healthy Wheat
Total collected: 3634
Valid images: 3634
After balancing: 1500

Processing Mildew
Total collected: 1105
Valid images: 1105
After balancing: 1105

Processing Yellow Rust
Total collected: 2974
Valid images: 2974
After balancing: 1500

Processing Brown Rust
Total collected: 4061
Valid images: 4061
After balancing: 1500

✅ FINAL DATASET READY: /kaggle/working/wheat_final


In [2]:
import os

base_dir = "/kaggle/working/wheat_final"  # change this

for split in ["train", "val", "test"]:
    split_path = os.path.join(base_dir, split)

    print(f"\n=== {split.upper()} ===")

    if not os.path.exists(split_path):
        print("❌ Missing:", split_path)
        continue

    for cls in os.listdir(split_path):
        class_path = os.path.join(split_path, cls)

        if os.path.isdir(class_path):
            count = len(os.listdir(class_path))
            print(f"{cls}: {count}")


=== TRAIN ===
yellow_rust: 1200
brown_rust: 1200
healthy: 1200
mildew: 884

=== VAL ===
yellow_rust: 150
brown_rust: 150
healthy: 150
mildew: 110

=== TEST ===
yellow_rust: 150
brown_rust: 150
healthy: 150
mildew: 111


In [3]:
import tensorflow as tf

Img=(224,224)
batch=32

train_ds=tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/wheat_final/train",
    image_size=Img,
    batch_size=batch
)
test_ds=tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/wheat_final/test",
    image_size=Img,
    batch_size=batch
)
val_ds=tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/wheat_final/val",
    image_size=Img,
    batch_size=batch
)
class_names=train_ds.class_names;
print(class_names)

2026-03-18 16:31:19.967093: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773851480.198912      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773851480.261620      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773851480.770205      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773851480.770237      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773851480.770240      24 computation_placer.cc:177] computation placer alr

Found 4484 files belonging to 4 classes.


I0000 00:00:1773851508.890739      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773851508.896538      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 561 files belonging to 4 classes.
Found 560 files belonging to 4 classes.
['brown_rust', 'healthy', 'mildew', 'yellow_rust']


In [4]:
AUTOTUNE=tf.data.AUTOTUNE
train_ds=train_ds.prefetch(AUTOTUNE)
val_ds=val_ds.prefetch(AUTOTUNE)

In [5]:
from tensorflow.keras import layers

data_augmentation=tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1)
        
    ]
)

In [6]:
from tensorflow import keras
wheat_model=keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)
wheat_model.trainable=False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
inputs = keras.Input(shape=(224,224,3))
x=data_augmentation(inputs)
x=keras.applications.mobilenet_v2.preprocess_input(x)
x=wheat_model(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.BatchNormalization()(x)
x=layers.Dense(128,activation="relu")(x)
x=layers.Dropout(0.3)(x)

outputs=layers.Dense(len(class_names),activation="softmax")(x)
model=keras.Model(inputs,outputs)

In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
early_stop = keras.callbacks.EarlyStopping(
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10


I0000 00:00:1773851519.077409      75 cuda_dnn.cc:529] Loaded cuDNN version 91002


141/141 ━━━━━━━━━━━━━━━━━━━━ 19s 75ms/step - accuracy: 0.6401 - loss: 1.0121 - val_accuracy: 0.8125 - val_loss: 0.4628
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 58ms/step - accuracy: 0.8088 - loss: 0.5087 - val_accuracy: 0.8589 - val_loss: 0.4000
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8370 - loss: 0.4497 - val_accuracy: 0.8786 - val_loss: 0.3586
Epoch 4/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8567 - loss: 0.3893 - val_accuracy: 0.8625 - val_loss: 0.3576
Epoch 5/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8646 - loss: 0.3569 - val_accuracy: 0.8643 - val_loss: 0.3488
Epoch 6/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.8707 - loss: 0.3346 - val_accuracy: 0.8732 - val_loss: 0.3640
Epoch 7/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.8836 - loss: 0.2938 - val_accuracy: 0.8732 - val_loss: 0.3606
Epoch 8/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.8975 - loss: 0.2654 - val_accuracy: 0.88

In [10]:
# unfreeze last layers only
for layer in wheat_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 22s 89ms/step - accuracy: 0.8359 - loss: 0.4707 - val_accuracy: 0.8875 - val_loss: 0.3505
Epoch 2/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 11s 80ms/step - accuracy: 0.8599 - loss: 0.4018 - val_accuracy: 0.8839 - val_loss: 0.3644
Epoch 3/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 11s 81ms/step - accuracy: 0.8663 - loss: 0.3445 - val_accuracy: 0.8893 - val_loss: 0.3518
Epoch 4/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 12s 82ms/step - accuracy: 0.8704 - loss: 0.3346 - val_accuracy: 0.8946 - val_loss: 0.3382
Epoch 5/5
141/141 ━━━━━━━━━━━━━━━━━━━━ 12s 83ms/step - accuracy: 0.8873 - loss: 0.2923 - val_accuracy: 0.8982 - val_loss: 0.3231


In [11]:
loss, acc = model.evaluate(test_ds)
print("Test Accuracy:", acc)

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.9204 - loss: 0.2514
Test Accuracy: 0.9073083996772766


In [12]:
model.save("/kaggle/working/wheat_model.keras")